The purpose of this notebook to generate the BM25 indexes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd '/content/drive/MyDrive/Colab Notebooks/Rag project'

/content/drive/MyDrive/Colab Notebooks/Rag project


In [ ]:
!pip install bm25s

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.5/75.5 kB 4.1 MB/s eta 0:00:00


In [ ]:
import bm25s
import json
import os
import pandas as pd

In [ ]:
# Create a directory named 'index' if it doesn't already exist
INDEX_DIR = "index"
os.makedirs("index", exist_ok=True)
print("Directory 'index' created successfully (or already exists).")

Directory 'index' created successfully (or already exists).


In [ ]:
# Method 1: If your JSON file is structured as a list of records or a simple dictionary table
df = pd.read_json('corpus_2026-10-09.jsonl')


In [ ]:
df.head(2)

,pubmed_id,title,keywords,journal,abstract,doi,accepted,published,authors
0,42853796,Performance of large language models in the Ch...,[],PLOS Digit Health,Large language models (LLMs) such as GPT-4 and...,10.1371/journal.pdig.0001773,2026-09-24,2026-10-09,"[Zong H, Wang J, Cha J, Wu R, Hu S, Zhao Y, Sh..."
1,42853748,EndoCADx: A Preference-Aligned Vision-Language...,[],IEEE J Biomed Health Inform,The subjective nature of descriptive endoscopi...,10.1109/JBHI.2026.3741523,None,2026-10-09,"[Hu S, Jiang Y, Xiong T, Xiao C, Jiang R, Dong..."


In [ ]:
ids = df['pubmed_id'].to_list()
docs = df['abstract'].to_list()
# sanity check & length of corpus -> we have pretty small one.
print(len(ids), len(docs))

100 100


#### tokenize corpus

In [ ]:
tokens = bm25s.tokenize(docs, stopwords = 'en')

Split strings:   0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
# tokens has two methods ids and vocabs -> lets see what we got
print(tokens.ids[:1])
print(tokens.vocab.keys())

[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 9, 15, 16, 17, 18, 19, 20, 21, 22, 3, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 22, 35, 36, 17, 37, 13, 9, 38, 39, 36, 40, 41, 42, 43, 44, 30, 31, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 37, 3, 13, 9, 38, 60, 53, 61, 62, 63, 35, 14, 64, 13, 9, 65, 66, 36, 67, 68, 69, 70, 71, 72, 73, 74, 75, 58, 56, 76, 77, 78, 79, 80, 35, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 3, 93, 60, 53, 61, 78, 94, 82, 95, 96, 97, 13, 9, 38, 98, 17, 99, 65, 20, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 58, 113, 65, 114, 3, 53, 62, 98, 4, 4, 4, 115, 116, 5, 117, 118, 119, 120, 121, 120, 122, 123, 124, 4, 115, 5, 117, 125, 126, 127, 128, 129, 130, 131, 132, 133, 110, 17, 134, 135, 136, 90, 137, 138, 139, 4, 125, 140, 141, 142, 143, 4, 144, 145, 146, 147, 148, 149, 150, 151, 82, 152, 153, 96, 61, 60, 3, 154, 155, 36, 13, 9, 65, 156, 157, 2, 158, 43, 34, 159, 160, 161, 162, 163, 164, 165, 82, 166, 167, 168, 16

In [ ]:
# building the retrieve

retriever = bm25s.BM25()
retriever.index(tokens)

DEBUG:bm25s:Building index from IDs objects


BM25S Count Tokens:   0%|          | 0/100 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
##### sanity check -> lets try to pull somethign out of the corpus using the key serach
# Tokenize our query first using the same tokenizer
query = "large language models"
query_tokens = bm25s.tokenize(query, stopwords='en')

# Retrieve top 3 results
results, scores = retriever.retrieve(query_tokens, corpus=docs, k=3)

# Print results and their corresponding BM25 scores
for i in range(results.shape[1]):
    print(f"Rank {i+1} (Score: {scores[0, i]:.4f}):")
    print(results[0, i])
    print("-" * 50)

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Rank 1 (Score: 0.5254):
Recent advances in ophthalmic foundation models have accelerated the application of artificial intelligence in ophthalmology, improving disease detection, progression assessment, and treatment evaluation. These advances are supported by the increasing availability of multimodal ophthalmic imaging data, including optical coherence tomography, color fundus photography, and slit lamp imaging, together with progress in self-supervised learning, vision-language modeling, and agent-based systems. This review summarizes publicly available multimodal ophthalmic imaging datasets and examines their foundational role in large-scale pretraining and cross-modal representation learning. We organize ophthalmic foundation models into vision foundation models, vision-language foundation models, and Large Language Model-based agentic systems, and discuss their core architectures, learning strategies, and representative applications. By synthesizing evidence across structural, vas

In [ ]:
# lets write the helper function to retire the info

def retrive(query, k = 3):
  query_tokens = bm25s.tokenize(query, stopwords='en')
  results, scores = retriever.retrieve(query_tokens, corpus=docs, k=5)
  return [(
      ids[i] , float(scores[0][j])) \
            for j ,i in enumerate(results[0].to_list()) \
  ]

In [ ]:
# save the values on a disk
retriever.save(INDEX_DIR)

# Fix: Use standard file writing to write the list of pubmed IDs
with open(os.path.join(INDEX_DIR, 'docs_ind.txt'), 'w') as f:
    f.write('\n'.join(map(str, ids)))